# 06 · CAR checks

Reads the output of `src/06_car.py` (run that first). Nothing here changes the data files.

1. Coverage: how many announcements have each CAR, and why some are missing
2. **Spot check against the WRDS Event Study tool** (the card's "done when")
3. **Average CAR path, treated vs control**: first look at H1 vs H2
4. Drift by surprise quintile: the long-short spread, treated vs control

Figures are saved to `output/figures/` and copied to `docs/assets/` for the project site.

In [ ]:
import os
import shutil
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
PROC = Path(os.environ.get("MEGACAP_PROC", ROOT / "data" / "processed"))
OUTPUT = ROOT / "output"
FIGS = OUTPUT / "figures"
ASSETS = ROOT / "docs" / "assets"

car = pd.read_parquet(PROC / "car.parquet")
sample = pd.read_parquet(PROC / "sample.parquet")
ar = pd.read_parquet(PROC / "ar_paths.parquet")
print(f"{len(car):,} announcements with CARs; sample has {len(sample):,} rows")

## 1. Coverage

In [ ]:
cols = ["car01", "car2_60", "car01_mkt", "car2_60_mkt"]
cov = pd.DataFrame({"has value": car[cols].notna().sum(),
                    "share": car[cols].notna().mean().map("{:.1%}".format)})
display(cov)
print("Missing car2_60 because the window runs past 2025-12-31:", (car["post_complete"] == 0).sum())
print("Missing market model (< 100 estimation days, mostly recent IPOs):", car["beta"].isna().sum())
last_full = car.loc[car["post_complete"] == 1, "reaction_day"].max()
print("Last reaction day with a full +2..+60 window:", pd.Timestamp(last_full).date())

## 2. Spot check against the WRDS Event Study tool

`06_car.py` picked 10 random events and wrote them to `output/car_spotcheck_input.txt` (one `PERMNO YYYYMMDD` per line).

1. WRDS → **Event Study by WRDS** → **U.S. Daily Event Study: Upload your own events**.
2. Upload `output/car_spotcheck_input.txt`.
3. Model: **Market Model**, **value-weighted** index.
4. Estimation parameters (WRDS places the estimation window relative to the *start* of the event window, so do **one run for days 0 to +60** and the notebook sums the daily abnormal returns):

   | Field | Value |
   |---|---|
   | Estimation Window | 221 |
   | Minimum Number of Valid Returns | 100 |
   | Gap | 29 (days −29 to −1, so estimation ends at −30) |
   | Event Window Start / End | 0 / 60 |

5. Variables: **UID** (or PERMNO), **EVTDATE**, **EVTTIME**, **ABRET**.
6. Output: CSV, YYYY-MM-DD. Save as `output/wrds_eventstudy_output.csv`.

The cell below sums WRDS's ABRET over days 0..+1 and +2..+60 and compares with ours. Differences of a few basis points can come from small convention differences. Large ones mean a real mismatch.

In [ ]:
ours = pd.read_csv(OUTPUT / "car_spotcheck.csv", parse_dates=["reaction_day"])
display(ours.round(4))

wrds_file = OUTPUT / "wrds_eventstudy_output.csv"
if wrds_file.exists():
    w = pd.read_csv(wrds_file)
    w.columns = w.columns.str.lower()
    # PERMNO: a permno column, else the first number in UID ("permno + event date")
    w["reaction_day"] = pd.to_datetime(w["evtdate"].astype(str), errors="coerce")
    if "permno" in w.columns:
        w["permno"] = pd.to_numeric(w["permno"], errors="coerce")
    else:
        # UID = PERMNO + event date, e.g. "17750_20180123" or "1775020180123"
        digits = w["uid"].astype(str).str.replace(r"\D", "", regex=True)
        ymd = w["reaction_day"].dt.strftime("%Y%m%d")
        w["permno"] = pd.to_numeric(
            [d[: -len(y)] if isinstance(y, str) and d.endswith(y) else d for d, y in zip(digits, ymd)],
            errors="coerce")
    w["evttime"] = pd.to_numeric(w["evttime"], errors="coerce")
    w["abret"] = pd.to_numeric(w["abret"], errors="coerce")
    agg = (w.assign(win=np.select([w["evttime"].between(0, 1), w["evttime"].between(2, 60)],
                                  ["car01_wrds", "car2_60_wrds"], "other"))
             .query("win != 'other'")
             .groupby(["permno", "reaction_day", "win"])["abret"].sum()
             .unstack("win").reset_index())
    cmp_ = ours.merge(agg, on=["permno", "reaction_day"], how="left")
    cmp_["diff_car01"] = cmp_["car01"] - cmp_["car01_wrds"]
    cmp_["diff_car2_60"] = cmp_["car2_60"] - cmp_["car2_60_wrds"]
    display(cmp_[["permno", "reaction_day", "car01", "car01_wrds", "diff_car01",
                  "car2_60", "car2_60_wrds", "diff_car2_60"]].round(5))
    print(f"Matched {cmp_['car01_wrds'].notna().sum()} of {len(cmp_)} events")
    print("Max abs difference CAR(0,+1):  ", round(cmp_["diff_car01"].abs().max(), 5))
    print("Max abs difference CAR(+2,+60):", round(cmp_["diff_car2_60"].abs().max(), 5))
else:
    print("No WRDS output yet. Save it as output/wrds_eventstudy_output.csv and re-run this cell.")

## 3. Average CAR path, treated vs control

Mean abnormal return per event day, cumulated from day −5, for the 1:1 matched sample with a complete +60 window. The two panels split firms by the sign of their surprise.

**What H1 (crowding out) looks like:** treated lines move *less* at day 0 than control lines, and the gap *closes* over the following 60 days (drift).

In [ ]:
BLUE, ORANGE = "#2a78d6", "#eb6834"          # categorical slots 1 and 2
INK, INK2, GRID, SURFACE = "#0b0b0b", "#52514e", "#e4e3df", "#fcfcfb"

s = sample[(sample["match_rank"] <= 1) & (sample["post_complete"] == 1) & sample["car01"].notna()]
s = s.assign(news=np.select([s["sue"] > 0, s["sue"] < 0], ["Beat (SUE > 0)", "Miss (SUE < 0)"], "meet"))
s = s[s["news"] != "meet"].merge(ar, on="accession")
day_cols = [c for c in ar.columns if c.startswith("ar_")]
days = np.array([int(c[3:]) for c in day_cols])

paths = {}
for (role, news), g in s.groupby(["role", "news"]):
    paths[(role, news)] = (100 * g[day_cols].mean().cumsum().values, len(g))

fig, axes = plt.subplots(1, 2, figsize=(10, 4.2), dpi=150, sharey=True)
fig.patch.set_facecolor(SURFACE)
for ax, news in zip(axes, ["Beat (SUE > 0)", "Miss (SUE < 0)"]):
    ax.set_facecolor(SURFACE)
    ends = {}
    for role, color in [("treated", BLUE), ("control", ORANGE)]:
        y, n = paths[(role, news)]
        ax.plot(days, y, color=color, linewidth=2, label=f"{role.title()} (n={n:,})")
        ends[role] = y[-1]
    # push end labels apart when the lines finish close together
    gap = 0.3
    t_y, c_y = ends["treated"], ends["control"]
    if abs(t_y - c_y) < gap:
        mid, sign = (t_y + c_y) / 2, (1 if t_y >= c_y else -1)
        t_y, c_y = mid + sign * gap / 2, mid - sign * gap / 2
    ax.text(days[-1] + 1, t_y, "Treated", color=INK, fontsize=9, va="center")
    ax.text(days[-1] + 1, c_y, "Control", color=INK, fontsize=9, va="center")
    ax.axhline(0, color=INK2, linewidth=0.8)
    ax.axvline(0, color=INK2, linewidth=0.8, linestyle=(0, (3, 3)))
    ax.set_title(news, loc="left", color=INK, fontsize=11)
    ax.set_xlabel("Trading days from reaction day", color=INK2)
    ax.grid(axis="y", color=GRID, linewidth=0.8)
    ax.set_axisbelow(True)
    for sp in ("top", "right", "left"):
        ax.spines[sp].set_visible(False)
    ax.spines["bottom"].set_color(GRID)
    ax.tick_params(colors=INK2, length=0)
    ax.set_xlim(days[0], days[-1] + 12)
    ax.set_xticks([0, 10, 20, 30, 40, 50, 60])
axes[0].set_ylabel("Cumulative abnormal return (%)", color=INK2)
axes[0].legend(frameon=False, fontsize=9, loc="lower left", labelcolor=INK)
fig.suptitle("Cumulative abnormal return around earnings, mega-cap days vs matched controls",
             x=0.01, ha="left", color=INK, fontsize=12)
fig.tight_layout()
FIGS.mkdir(parents=True, exist_ok=True)
fig.savefig(FIGS / "car_paths.png", facecolor=SURFACE)
plt.show()

## 4. Drift by surprise quintile

SUE quintiles are formed within each calendar quarter over all announcements. The **long-short** row (Q5 − Q1) is the drift a strategy would capture. H1 predicts the treated long-short CAR(+2,+60) is **larger** than the control one, and the treated CAR(0,+1) spread is **smaller**.

In [ ]:
q = car.merge(pd.read_parquet(PROC / "sue.parquet")[["accession", "sue", "quarter"]], on="accession")
q["sue_q"] = q.groupby("quarter")["sue"].transform(
    lambda x: pd.qcut(x.rank(method="first"), 5, labels=False) + 1)
s2 = sample[sample["match_rank"] <= 1].merge(q[["accession", "sue_q"]], on="accession")
s2 = s2[s2["post_complete"] == 1]

tab = (s2.groupby(["role", "sue_q"])[["car01", "car2_60"]].mean() * 100).unstack("role")
tab.loc["Q5 - Q1"] = tab.loc[5] - tab.loc[1]
tab.index = [f"Q{i}" if not isinstance(i, str) else i for i in tab.index]
display(tab.round(2))
print("Descriptive only: no controls and no standard errors. The regressions (E1, E2) come next.")

## Copy figures to the project site

In [ ]:
ASSETS.mkdir(parents=True, exist_ok=True)
shutil.copy(FIGS / "car_paths.png", ASSETS / "car_paths.png")
print("Copied car_paths.png to docs/assets/")